# 01 - Data exploration and leakage checks

Produces the midterm report's **dataset statistics** and **leakage checks** (guidelines §3.2).

Prerequisites: CIFAKE unzipped into `data/raw`, and `python -m src.dataset --make-split` already run.
Run the notebook from the repo root or from `notebooks/`.

**Status:** the setup cell is implemented. Sections 1-4 are TODO for the team.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config, repo_path
from src.dataset import LABELS, label_from_path, list_images, pool_from_path, read_split

cfg = load_config(ROOT / "configs/default.yaml")
RAW = repo_path(cfg["data"]["raw_dir"])
SPLITS = repo_path(cfg["data"]["splits_dir"])
train_paths = read_split(SPLITS / "train.txt")
val_paths = read_split(SPLITS / "val.txt")
test_paths = list_images(RAW, "test")
print(f"train={len(train_paths)} val={len(val_paths)} test={len(test_paths)}")

## 1. Class counts per split
Expected: train 45,000 / 45,000, val 5,000 / 5,000, test 10,000 / 10,000 (REAL / FAKE).

In [ ]:
# TODO(team): count REAL/FAKE per split with label_from_path; show as a small table (pandas optional).

## 2. Sample grid: 8 REAL and 8 FAKE
Use a fixed `random.Random(cfg["seed"])` so the figure is reproducible. Save it to `results/figures/` for the report.

In [ ]:
# TODO(team): 2x8 matplotlib grid from val_paths; PIL.Image.open(RAW / p); interpolation="nearest".

## 3. Leakage checks
- No path appears in both `train.txt` and `val.txt`.
- No train/val path is under the test folder (`pool_from_path(p) == "test"`).
- Optional, stronger: no **duplicate image content** across splits (hash the file bytes, e.g. `hashlib.md5`). Path checks cannot catch the same image saved twice.

In [ ]:
# TODO(team): assert the checks above and print a one-line PASS/FAIL for each (quote these in the midterm).

## 4. Confounder check: JPEG quantization tables by class
CIFAKE images are stored as JPEG. If REAL and FAKE were saved with **different quantization tables / quality**,
a classifier can separate them by compression history alone, and the JPEG corruptions would be testing that, not diffusion artifacts.
PIL exposes the tables as `Image.open(path).quantization` (a dict of 64-value tables).

In [ ]:
# TODO(team): sample ~500 files per class; collect a hashable form of .quantization; compare the
# distribution of distinct tables between REAL and FAKE. Record the finding for Data / Limitations.